# The analysis: one snapshot, one run, one covariance

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.study`

**Modules covered** `study.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The analysis is not a method: it is the one place a snapshot's run and the readings taken off it are assembled. Nine call sites used to do that for themselves - the five entry points, the memo, the workbook, the grid's own report and the acceptance fixture - and an assembly is where a convention lives, which is how the risk budget came to be read against a covariance taken over a different set of months from the one the cells were built on. The entry point prints the shape of the run rather than a result: the numbers live in the modules behind it, each with its own report.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `study.py::analyse` traces to a composition of this effort, not a method: the one place the run and the covariance every reading of it is taken against are assembled, so that two reports cannot be two objects
- `study.py::main` traces to a composition of this effort, not a method: the one place the run and the covariance every reading of it is taken against are assembled, so that two reports cannot be two objects
- `study.py::risk_budgets` traces to a composition of this effort, not a method: the one place the run and the covariance every reading of it is taken against are assembled, so that two reports cannot be two objects
- `study.py::traded_months` traces to a composition of this effort, not a method: the one place the run and the covariance every reading of it is taken against are assembled, so that two reports cannot be two objects

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`study.py`**

The analysis: one snapshot, analysed once, with one covariance behind every reading of it.

**Why it exists.** The grid, the benchmark, the comparison table, the attribution and the risk budget
are five readings of one run, and each entry point that prints them used to assemble that run for
itself. Nine call sites made nine assemblies, and an assembly is where a convention lives: the sample
covariance the risk budget decomposes was read over the panel's 190 returns at one call site and over
the 131 traded months at another, so two published documents stated different numbers for the same
book, and nothing could catch it because no module owned the quantity they both claimed.

**What it owns.** Which months a reading is taken over, which covariance it is taken against, and the
order the run is assembled in. The covariance is the sample covariance over the traded months, taken
from the estimator module rather than re-derived here: the risk budget describes the books the run
produced, so the months those books were held are the months their covariance is read over. The
estimator is named in the report this module prints, because a budget read against a different
covariance from the one the cell was built on is a statement about two objects and not one.

**What it deliberately does not own.** The mandate. The window, the universe, the policy weights and
the constraint set are this build's decisions for one panel and one mandate and stay in the modules
that declare them; a consumer with a different mandate supplies its own document. Nor does it own the
modules' own vocabularies: the count series, the spanning directions and the headline decomposition
belong to the one caller that reports them, and pulling them in here would widen this interface
without giving a single further caller anything.

**Where it sits.** Above the layers and below the output surface: `reporting/` reads the analysis and
nothing here reads `reporting/`. The entry points import this module inside their own functions rather
than at module level, because this module imports the layers those entry points live in - the same
deferral the comparison table already uses to reach the loader.

## 3. The data contract it consumes, and the as-of rule

The analysis consumes the document the loader returns and adds no contract of its own: the table contract, the as-of rule and the panel window belong to `data/`, and this module inherits them by reading what the loader handed it. What it does add is a constraint on the readings - every one of them is taken over the months the run traded, and against one sample covariance over those months - so a report naming a different window is describing a different object rather than a second view of this one.

## 4. The worked example on small numbers, with the identity checked

The analysis's own identity is that its window is the run's window rather than a second cut of the calendar, and that it refuses a grid with no run instead of reading an empty frame as a window of zero months. The cell plants both cases; the covariance itself is checked where it belongs, in the acceptance fixture, which holds the analysis's covariance against the estimator module's own.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import pandas as pd

from portfolio_workbench import study

# The window every reading is taken over is read off the run rather than re-cut, so a grid that traded
# three months is analysed over those three and not over whatever the calendar would have offered.
planted = {"results": [{"traded": pd.PeriodIndex(["2020-01", "2020-02", "2020-03"], freq="M")}]}
assert [str(month) for month in study.traded_months(planted)] == ["2020-01", "2020-02", "2020-03"]

# A grid that produced no run has no window, and the analysis refuses rather than reading the absence
# as a window of zero months.
try:
    study.traded_months({"results": []})
except ValueError as refusal:
    print("refused:", refusal)
else:
    raise AssertionError("a grid with no run should not yield a window")

refused: the grid produced no run, so the analysis has no traded months to read over


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.compare import registry
from portfolio_workbench.data import universe

print(f"sleeves {len(universe.TICKERS)}: {', '.join(universe.TICKERS)}")
print(f"declared panel window {universe.WINDOW_START}..{universe.WINDOW_END}")
print(f"cells {len(registry.CELLS)} over {registry.PRE_REGISTERED} pre-registered runs")
print("covariance every reading is taken against: the sample covariance over the traded months")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
sleeves 11: IWDA.AS, IMEU.AS, XACT-NORDEN.ST, IBGL.AS, IEGE.AS, IEAC.AS, IHYG.L, IBCI.AS, 4GLD.DE, IWDP.AS, XEON.DE
declared panel window 2010-09..2026-08
cells 16 over 20 pre-registered runs
covariance every reading is taken against: the sample covariance over the traded months


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.study"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[table] analysis: 191 panel months, 131 traded (2015-09..2026-07), 20 runs and 0 cut
[table] covariance: sample, over the traded months, 11 sleeves; the estimator every risk budget below is read against
[table] readings: 20 table rows over 16 distinct cells, 20 attributed books, 20 budgeted books
[table] WARNING recomputed-TR divergence: XACT-NORDEN.ST adjusted close implies +424.74% cumulative, close plus distributions implies +640.27%; the series is used as published and the divergence travels with it
[table] WARNING thin liquidity: IBGL.AS reports EUR 0.94bn, below the EUR 1.35bn floor; the flat per-side cost rate understates what trading this sleeve costs
[table] WARNING thin liquidity: IEGE.AS reports EUR 1.30bn, below the EUR 1.35bn floor; the flat per-side cost rate understates what trading this sleeve costs
[table] WARNING liquidity not screened: 4GLD.DE (n/a), IMEU.AS (n/a), XEON.DE (n/a) publish no fund size, so the floor cannot be applied to them
[table] WARNING unexpected d

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The analysis is what makes two reports comparable: the window, the estimator and the book set their readings share are assembled in one place and stated by its entry point, so a reader holding the comparison table and the risk budget is holding two readings of one run. A reader must not read it as a result, because it computes nothing the modules behind it do not, and must not read it as the mandate either - the panel window, the universe, the policy weights and the constraint set are this build's decisions for one panel, and a consumer with a different mandate supplies its own document.

## 7. What this module does not establish

Nothing here establishes anything about the panel, the methods or the cost convention: those are the modules' own subjects, and each states what it does not establish in its own notebook. It does not establish that the traded months are the right months to trade, which is the evaluation design's claim rather than a property of this module, and it makes no claim about the consumer boundary, which deliberately carries none of these decisions.